In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelterCRUD

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "aac123"

# Connect to database via CRUD Module
db = AnimalShelterCRUD(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # Branding/Header (logo clickable + your name)
    html.Div([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '120px', 'marginBottom': '10px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H2('Grazioso Salvare Rescue Dashboard', style={'margin': '0'}),
        html.H5('Developed by Rowvin Dizon', style={'marginTop': '6px'})
    ], style={'textAlign': 'center'}),

    html.Hr(),

    html.Div(
        #FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        children=[
            html.H4("Rescue Type Filter", style={'textAlign': 'center'}),

            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Reset (All)', 'value': 'reset'},
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
                ],
                value='reset',
                labelStyle={'display': 'block', 'margin': '6px 0'}
            )
        ],
        style={'width': '40%', 'margin': '0 auto'}
    ),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),

        #FIXME: Set up the features for your interactive data table to make it user-friendly for your client
        #If you completed the Module Six Assignment, you can copy in the code you created here 
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        cell_selectable=True,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '6px', 'whiteSpace': 'normal'},
        style_header={'fontWeight': 'bold'},
    ),

    html.Br(),
    html.Hr(),

    # Side-by-side chart + map
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6'),
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

def build_query(filter_type: str) -> dict:
    base = {"animal_type": "Dog"}

    if filter_type == "water":
        return {
            "$and": [
                base,
                {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}},
                {"sex_upon_outcome": "Intact Female"},
                {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
            ]
        }

    if filter_type == "mountain":
        return {
            "$and": [
                base,
                {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                                  "Siberian Husky", "Rottweiler"]}},
                {"sex_upon_outcome": "Intact Male"},
                {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
            ]
        }

    if filter_type == "disaster":
        return {
            "$and": [
                base,
                {"breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                                  "Bloodhound", "Rottweiler"]}},
                {"sex_upon_outcome": "Intact Male"},
                {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
            ]
        }

    return {}  # reset

    
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    ## FIX ME Add code to filter interactive data table with MongoDB queries
    query = build_query(filter_type)

    try:
        results = db.read(query)
        dff = pd.DataFrame.from_records(results)
        if "_id" in dff.columns:
            dff.drop(columns=["_id"], inplace=True)

        # Reset selection so the map always has a valid row
        return dff.to_dict("records"), [0]

    except Exception as e:
        print("Dashboard query error:", e)
        return df.to_dict("records"), [0]

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):
    ###FIX ME ####
    if viewData is None or len(viewData) == 0:
        return html.Div("No data to display.")

    dff = pd.DataFrame.from_dict(viewData)

    breed_counts = dff["breed"].value_counts()
    top = breed_counts.head(10)
    other = breed_counts.iloc[10:].sum()
    if other > 0:
        top["Other"] = other

    pie_df = top.reset_index()
    pie_df.columns = ["breed", "count"]

    fig = px.pie(pie_df, names="breed", values="count", title="Breed Distribution (Top 10)")

    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available for the map.")

    dff = pd.DataFrame.from_dict(viewData)

    # Default to first row if none selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    try:
        lat = float(dff.loc[row, "location_lat"])
        lon = float(dff.loc[row, "location_long"])
    except (TypeError, ValueError):
        return html.Div("Selected record has invalid location data.")

    breed = dff.loc[row, "breed"] if "breed" in dff.columns else "Unknown"
    name = dff.loc[row, "name"] if "name" in dff.columns else "Unknown"

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(name)
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://alphatemple-fiberweekend-3000.codio.io/proxy/8050/
